In [1]:
import pandas as pd
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

import sys
from pathlib import Path
import os

from sklearn.model_selection import train_test_split

In [2]:
# Semilla para reproducibilidad de los experimentos
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [3]:
# Me aseguro de que el directorio raíz del proyecto esté en el sys.path
project_root = Path(os.path.abspath("")).parent

# Añado el directorio raíz al sys.path si no está ya presente
if project_root not in sys.path:
    sys.path.append(str(project_root))

In [4]:
# Importo las funciones de configuración
from src.config import interim_data_dir, processed_data_dir, load_config



Current working directory: /home/jorge/development/WasteImageReconstructionDL/notebooks
Loading configuration from /home/jorge/development/WasteImageReconstructionDL/src/config.yaml


In [5]:
# Cargo la configuración 
config = load_config()

Loading configuration from /home/jorge/development/WasteImageReconstructionDL/src/config.yaml


In [6]:
# Ruta de los datos
base_dir = interim_data_dir() / 'dataset_scaled_2'

In [7]:
# Inicializamos listas para almacenar rutas de imágenes y sus etiquetas
image_paths = []
labels = []

In [8]:
# Recorremos cada subdirectorio (que representa una clase)
for class_name in os.listdir(base_dir):
    class_dir = os.path.join(base_dir, class_name)
    if os.path.isdir(class_dir):
        # Recorremos cada imagen en el subdirectorio
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            image_paths.append(img_path)
            labels.append(class_name)

In [9]:
# Ahora image_paths contiene todas las rutas de las imágenes y labels sus etiquetas correspondientes
image_paths[:5], labels[:5]

(['/home/jorge/development/WasteImageReconstructionDL/data/interim/dataset_scaled_2/bean/e100673e-b9d1-4328-a7bd-ebbd0075a131.png',
  '/home/jorge/development/WasteImageReconstructionDL/data/interim/dataset_scaled_2/bean/fc989cc4-7509-4f4b-86e6-0a3467ae0662.png',
  '/home/jorge/development/WasteImageReconstructionDL/data/interim/dataset_scaled_2/bean/6e56a1bc-5891-4b8d-be78-de89b8bf3c99.png',
  '/home/jorge/development/WasteImageReconstructionDL/data/interim/dataset_scaled_2/bean/46fcb418-a4df-4f23-9687-f271c253a50f.png',
  '/home/jorge/development/WasteImageReconstructionDL/data/interim/dataset_scaled_2/bean/48f7df55-99f4-491a-bdb5-028b75686ef3.png'],
 ['bean', 'bean', 'bean', 'bean', 'bean'])

In [10]:
# División del dataset en entrenamiento y validación con estratificación
# train_paths, val_paths, train_labels, val_labels = train_test_split(
#     image_paths, labels, test_size=0.2, stratify=labels, random_state=42
# )

# Ahora, train_paths y val_paths contienen las rutas de las imágenes para los conjuntos de entrenamiento y validación

> En este caso dividiré en los tres subconjuntos: entrenamiento, validación y testing.

In [12]:
# Dividiré inicialmente en entrenamiento y conjunto combinado de validación/test
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths, labels, test_size=0.3, stratify=labels, random_state=42
)

In [13]:
# Dividiré el conjunto combinado en validación y test
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

## Verificación

In [14]:
from collections import Counter

# Contar la cantidad de imágenes por clase en cada conjunto
train_distribution = Counter(train_labels)
val_distribution = Counter(val_labels)
test_distribution = Counter(test_labels)

print("Distribución en entrenamiento:", train_distribution)
print("Distribución en validación:", val_distribution)
print("Distribución en test:", test_distribution)


Distribución en entrenamiento: Counter({'peanut': 280, 'soy': 280, 'bean': 280, 'fallow': 280, 'sugarcane': 280, 'corn': 280, 'cotton': 280, 'wheat': 280})
Distribución en validación: Counter({'peanut': 60, 'soy': 60, 'fallow': 60, 'wheat': 60, 'cotton': 60, 'bean': 60, 'corn': 60, 'sugarcane': 60})
Distribución en test: Counter({'corn': 60, 'soy': 60, 'wheat': 60, 'sugarcane': 60, 'peanut': 60, 'bean': 60, 'cotton': 60, 'fallow': 60})


Guardamos los archivos de entrenamiento y validación

In [16]:
# Ruta de los datos
final_data_dir = processed_data_dir() 
os.makedirs(final_data_dir, exist_ok=True)

In [17]:
# Crear DataFrames
train_df = pd.DataFrame({'image_path': train_paths, 'label': train_labels})
val_df = pd.DataFrame({'image_path': val_paths, 'label': val_labels})
test_df = pd.DataFrame({'image_path': test_paths, 'label': test_labels}) 

In [18]:
# Guardar en final_data_dir/train.csv y final_data_dir/val.csv
train_df.to_csv(os.path.join(final_data_dir, 'train.csv'), index=False)
val_df.to_csv(os.path.join(final_data_dir, 'val.csv'), index=False)
test_df.to_csv(os.path.join(final_data_dir, 'test.csv'), index=False)